## Experiment - Diffusion - MNIST
- This notebook generates some figures used for the manuscript. 

## MNIST Data

In [ ]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# Load the dataset
mnist_ds = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transforms.ToTensor()
)
targets = mnist_ds.targets

# Loop through digits 0-9
for digit in range(10):
    # 1. Find ALL indices for the current digit
    all_idx = (targets == digit).nonzero(as_tuple=True)[0]
    
    # 2. Shuffle those indices randomly
    shuffled_idx = all_idx[torch.randperm(len(all_idx))]
    
    # 3. Now grab the first 64 from the randomized list
    idx = shuffled_idx[:64]
    
    # Stack, grid, and plot
    images = torch.stack([mnist_ds[i][0] for i in idx])
    grid = vutils.make_grid(images, nrow=8, padding=2)
    grid_np = grid.permute(1, 2, 0).numpy()
    
    plt.figure(figsize=(5, 5))
    plt.title(f"Class: {digit}", fontsize=14, fontweight='bold')
    plt.imshow(grid_np, cmap="gray")
    plt.axis("off")
    plt.show()

# Manuscript Figures
- Change experiment_root to the path of code output

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl

# -----------------------------
# GLOBAL STYLE
# -----------------------------

# Set font to Arial (sans-serif)
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]

# Set font sizes for publication
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 20
mpl.rcParams["axes.labelsize"] = 20
mpl.rcParams["legend.fontsize"] = 18
mpl.rcParams["legend.title_fontsize"] = 16
mpl.rcParams["xtick.labelsize"] = 18
mpl.rcParams["ytick.labelsize"] = 18

# Save
save = True
clean = False

# 1. Point this to your specific experiment root folder
EXPERIMENT_ROOT = Path(
    "logs/FINAL_mnist_4grid_fixed_FID_seed=12345_T=200_total=300_epochs=100_reps=1_Tdiff=400_cosine_bs=100_lr=0.0002_ch=64_train=fresh_balreal=1"
)
runs_dir = EXPERIMENT_ROOT / "runs"

# 2. Gather the data
# Dictionary structure:
# { alpha_value: {'fids': [array_rep1, array_rep2, ...]} }
plot_data = {}

for npz_path in runs_dir.rglob("metrics.npz"):
    data = np.load(npz_path)

    alpha = float(data["alpha_real"])
    fid = data["fid_history"]

    if alpha not in plot_data:
        plot_data[alpha] = {"fids": []}

    plot_data[alpha]["fids"].append(fid)

# Sort alphas once
alphas = sorted(plot_data.keys())

# remove alpha = 0.75 for now
alphas = [alpha for alpha in alphas if not np.isclose(alpha, 0.75)]
# -----------------------------
# VIRIDIS COLOR SCHEME
# -----------------------------

viridis_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(alphas)))
alpha_to_color = dict(zip(alphas, viridis_colors))

# 3. Plot the data
plt.figure(figsize=(10, 6))

for alpha in alphas:
    fids_list = plot_data[alpha]["fids"]

    # In case runs have slightly different lengths, truncate to shortest
    min_len = min(len(fid) for fid in fids_list)
    fids = np.array([fid[:min_len] for fid in fids_list])

    # Remove/ignore invalid values before logging
    # log is only defined for positive values
    fids = np.where(fids > 0, fids, np.nan)

    # Log-transform the actual FID values
    log_fids = np.log(fids)

    # Calculate mean and standard deviation across reps in log-space
    log_fid_mean = np.nanmean(log_fids, axis=0)
    log_fid_std = np.nanstd(log_fids, axis=0)

    # Iteration axis: 1, 2, ..., T_steps
    iterations = np.arange(1, len(log_fid_mean) + 1)

    # Plot mean log(FID)
    plt.plot(
        iterations,
        log_fid_mean,
        label=f"$\\alpha$ = {alpha:.2f}",
        linewidth=2,
        color=alpha_to_color[alpha]
    )

    # Add shaded region for standard deviation if there are multiple reps
    if log_fids.shape[0] > 1:
        plt.fill_between(
            iterations,
            log_fid_mean - log_fid_std,
            log_fid_mean + log_fid_std,
            color=alpha_to_color[alpha],
            alpha=0.2
        )

# 4. Formatting
plt.xlabel("Iteration")
plt.ylabel(r"$\log(\mathrm{FID})$")
# plt.title(r"CRT: Evaluation FID")
plt.legend(
    loc="lower right",
    bbox_to_anchor=(1.0, 0.55))
plt.tight_layout()

if save:
    fig_dir = EXPERIMENT_ROOT / "figures"
    fig_dir.mkdir(exist_ok=True)

    # --- FULL VERSION ---
    plt.savefig(fig_dir / "log_fid_vs_iteration.pdf", bbox_inches="tight")
    plt.savefig(fig_dir / "log_fid_vs_iteration.png", dpi=300, bbox_inches="tight")

    # --- CLEAN VERSION (remove labels/legend/title) ---
    if clean:
        plt.title("")
        plt.xlabel("")
        plt.ylabel("")

        leg = plt.gca().get_legend()
        if leg is not None:
            leg.remove()

        plt.savefig(fig_dir / "log_fid_vs_iteration_clean.pdf", bbox_inches="tight")
        plt.savefig(fig_dir / "log_fid_vs_iteration_clean.png", dpi=300, bbox_inches="tight")

plt.show()